In [17]:
import pandas as pd
import psycopg2
from psycopg2 import sql

# Database Connection
DB_CONFIG = {
    'host': 'localhost',
    'port': 5432,
    'database': 'logitrack_db',
    'user': 'postgres',
    'password': 'killua'  # Your password
}

# Connect
connection = psycopg2.connect(**DB_CONFIG)
cursor = connection.cursor()

print("✓ Connected to PostgreSQL - logitrack_db")
print("Ready to run queries...\n")

✓ Connected to PostgreSQL - logitrack_db
Ready to run queries...



In [18]:
query = """
-- Query 1: Overall Delivery Performance
SELECT 
    COUNT(*) as total_orders,
    SUM(CASE WHEN late_delivery_risk = 1 THEN 1 ELSE 0 END) as late_orders,
    SUM(CASE WHEN late_delivery_risk = 0 THEN 1 ELSE 0 END) as on_time_orders,
    ROUND(100.0 * SUM(CASE WHEN late_delivery_risk = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) as late_rate_percent,
    ROUND(AVG(days_for_shipping_real - days_for_shipment_scheduled), 2) as avg_delay_days,
    ROUND(SUM(sales_per_customer)::numeric, 2) as total_revenue,
    ROUND(AVG(sales_per_customer)::numeric, 2) as avg_order_value
FROM orders;
"""

cursor.execute(query)
df_result = pd.DataFrame(cursor.fetchall(), columns=[desc[0] for desc in cursor.description])

print("="*70)
print("QUERY 1: OVERALL DELIVERY PERFORMANCE")
print("="*70)
print(df_result.to_string(index=False))
print()

QUERY 1: OVERALL DELIVERY PERFORMANCE
 total_orders late_orders on_time_orders late_rate_percent avg_delay_days total_revenue avg_order_value
            0        None           None              None           None          None            None



In [19]:
query = """
-- Query 2: Shipping Mode Performance
SELECT 
    shipping_mode,
    COUNT(*) as order_count,
    SUM(CASE WHEN late_delivery_risk = 1 THEN 1 ELSE 0 END) as late_count,
    ROUND(100.0 * SUM(CASE WHEN late_delivery_risk = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) as late_rate_percent,
    ROUND(AVG(days_for_shipping_real - days_for_shipment_scheduled), 2) as avg_delay_days,
    ROUND(MIN(days_for_shipping_real - days_for_shipment_scheduled), 2) as min_delay_days,
    ROUND(MAX(days_for_shipping_real - days_for_shipment_scheduled), 2) as max_delay_days,
    ROUND(SUM(sales_per_customer)::numeric, 2) as total_revenue
FROM orders
GROUP BY shipping_mode
ORDER BY late_rate_percent DESC;
"""

cursor.execute(query)
df_result = pd.DataFrame(cursor.fetchall(), columns=[desc[0] for desc in cursor.description])

print("="*70)
print("QUERY 2: SHIPPING MODE PERFORMANCE")
print("="*70)
print(df_result.to_string(index=False))
print()

QUERY 2: SHIPPING MODE PERFORMANCE
Empty DataFrame
Columns: [shipping_mode, order_count, late_count, late_rate_percent, avg_delay_days, min_delay_days, max_delay_days, total_revenue]
Index: []



In [20]:
query = """
-- Query 3: Top 10 States by Late Delivery Rate
SELECT 
    customer_state,
    COUNT(*) as order_count,
    SUM(CASE WHEN late_delivery_risk = 1 THEN 1 ELSE 0 END) as late_count,
    ROUND(100.0 * SUM(CASE WHEN late_delivery_risk = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) as late_rate_percent,
    ROUND(AVG(days_for_shipping_real - days_for_shipment_scheduled), 2) as avg_delay_days
FROM orders
GROUP BY customer_state
HAVING COUNT(*) > 100
ORDER BY late_rate_percent DESC
LIMIT 10;
"""

cursor.execute(query)
df_result = pd.DataFrame(cursor.fetchall(), columns=[desc[0] for desc in cursor.description])

print("="*70)
print("QUERY 3: TOP 10 STATES BY LATE DELIVERY RATE")
print("="*70)
print(df_result.to_string(index=False))
print()

QUERY 3: TOP 10 STATES BY LATE DELIVERY RATE
Empty DataFrame
Columns: [customer_state, order_count, late_count, late_rate_percent, avg_delay_days]
Index: []



In [21]:
query = """
-- Query 4: Product Category Performance
SELECT 
    category_name,
    COUNT(*) as order_count,
    SUM(CASE WHEN late_delivery_risk = 1 THEN 1 ELSE 0 END) as late_count,
    ROUND(100.0 * SUM(CASE WHEN late_delivery_risk = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) as late_rate_percent,
    ROUND(AVG(days_for_shipping_real - days_for_shipment_scheduled), 2) as avg_delay_days,
    ROUND(AVG(sales_per_customer)::numeric, 2) as avg_order_value
FROM orders
GROUP BY category_name
ORDER BY order_count DESC
LIMIT 15;
"""

cursor.execute(query)
df_result = pd.DataFrame(cursor.fetchall(), columns=[desc[0] for desc in cursor.description])

print("="*70)
print("QUERY 4: PRODUCT CATEGORY PERFORMANCE (Top 15)")
print("="*70)
print(df_result.to_string(index=False))
print()

QUERY 4: PRODUCT CATEGORY PERFORMANCE (Top 15)
Empty DataFrame
Columns: [category_name, order_count, late_count, late_rate_percent, avg_delay_days, avg_order_value]
Index: []



In [22]:
query = """
-- Query 5: Customer Segment Performance
SELECT 
    customer_segment,
    COUNT(*) as order_count,
    SUM(CASE WHEN late_delivery_risk = 1 THEN 1 ELSE 0 END) as late_count,
    ROUND(100.0 * SUM(CASE WHEN late_delivery_risk = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) as late_rate_percent,
    ROUND(AVG(days_for_shipping_real - days_for_shipment_scheduled), 2) as avg_delay_days,
    ROUND(SUM(sales_per_customer)::numeric, 2) as total_revenue,
    ROUND(AVG(sales_per_customer)::numeric, 2) as avg_order_value
FROM orders
GROUP BY customer_segment
ORDER BY order_count DESC;
"""

cursor.execute(query)
df_result = pd.DataFrame(cursor.fetchall(), columns=[desc[0] for desc in cursor.description])

print("="*70)
print("QUERY 5: CUSTOMER SEGMENT PERFORMANCE")
print("="*70)
print(df_result.to_string(index=False))
print()

QUERY 5: CUSTOMER SEGMENT PERFORMANCE
Empty DataFrame
Columns: [customer_segment, order_count, late_count, late_rate_percent, avg_delay_days, total_revenue, avg_order_value]
Index: []



In [23]:
query = """
-- Query 6: Shipping Mode Performance by Region (Top States)
SELECT 
    shipping_mode,
    customer_state,
    COUNT(*) as order_count,
    ROUND(100.0 * SUM(CASE WHEN late_delivery_risk = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) as late_rate_percent,
    ROUND(AVG(days_for_shipping_real - days_for_shipment_scheduled), 2) as avg_delay_days
FROM orders
WHERE customer_state IN (
    SELECT customer_state 
    FROM orders 
    GROUP BY customer_state 
    ORDER BY COUNT(*) DESC 
    LIMIT 5
)
GROUP BY shipping_mode, customer_state
ORDER BY customer_state, late_rate_percent DESC;
"""

cursor.execute(query)
df_result = pd.DataFrame(cursor.fetchall(), columns=[desc[0] for desc in cursor.description])

print("="*70)
print("QUERY 6: SHIPPING MODE BY REGION (Top 5 States)")
print("="*70)
print(df_result.to_string(index=False))
print()

QUERY 6: SHIPPING MODE BY REGION (Top 5 States)
Empty DataFrame
Columns: [shipping_mode, customer_state, order_count, late_rate_percent, avg_delay_days]
Index: []



In [24]:
query = """
-- Query 7: Delivery Status Distribution
SELECT 
    delivery_status,
    COUNT(*) as order_count,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) as percentage,
    ROUND(AVG(days_for_shipping_real - days_for_shipment_scheduled), 2) as avg_delay_days
FROM orders
GROUP BY delivery_status
ORDER BY order_count DESC;
"""

cursor.execute(query)
df_result = pd.DataFrame(cursor.fetchall(), columns=[desc[0] for desc in cursor.description])

print("="*70)
print("QUERY 7: DELIVERY STATUS DISTRIBUTION")
print("="*70)
print(df_result.to_string(index=False))
print()

QUERY 7: DELIVERY STATUS DISTRIBUTION
Empty DataFrame
Columns: [delivery_status, order_count, percentage, avg_delay_days]
Index: []



In [25]:
query = """
-- Query 8: Shipping Type Distribution & Performance
SELECT 
    type,
    COUNT(*) as order_count,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) as percentage,
    SUM(CASE WHEN late_delivery_risk = 1 THEN 1 ELSE 0 END) as late_count,
    ROUND(100.0 * SUM(CASE WHEN late_delivery_risk = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) as late_rate_percent,
    ROUND(AVG(days_for_shipping_real - days_for_shipment_scheduled), 2) as avg_delay_days
FROM orders
GROUP BY type
ORDER BY order_count DESC;
"""

cursor.execute(query)
df_result = pd.DataFrame(cursor.fetchall(), columns=[desc[0] for desc in cursor.description])

print("="*70)
print("QUERY 8: SHIPPING TYPE DISTRIBUTION & PERFORMANCE")
print("="*70)
print(df_result.to_string(index=False))
print()

QUERY 8: SHIPPING TYPE DISTRIBUTION & PERFORMANCE
Empty DataFrame
Columns: [type, order_count, percentage, late_count, late_rate_percent, avg_delay_days]
Index: []



In [26]:
query = """
-- Query 9: Scheduled vs Actual Shipping Days Comparison
SELECT 
    days_for_shipment_scheduled,
    COUNT(*) as order_count,
    ROUND(AVG(days_for_shipping_real), 2) as avg_actual_days,
    ROUND(AVG(days_for_shipping_real - days_for_shipment_scheduled), 2) as avg_delay_days,
    ROUND(100.0 * SUM(CASE WHEN late_delivery_risk = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) as late_rate_percent
FROM orders
GROUP BY days_for_shipment_scheduled
ORDER BY days_for_shipment_scheduled;
"""

cursor.execute(query)
df_result = pd.DataFrame(cursor.fetchall(), columns=[desc[0] for desc in cursor.description])

print("="*70)
print("QUERY 9: SCHEDULED vs ACTUAL SHIPPING DAYS")
print("="*70)
print(df_result.to_string(index=False))
print()

QUERY 9: SCHEDULED vs ACTUAL SHIPPING DAYS
Empty DataFrame
Columns: [days_for_shipment_scheduled, order_count, avg_actual_days, avg_delay_days, late_rate_percent]
Index: []



In [27]:
query = """
-- Query 10: Revenue Impact by Delivery Status
SELECT 
    CASE 
        WHEN late_delivery_risk = 1 THEN 'Late'
        ELSE 'On-Time'
    END as delivery_status,
    COUNT(*) as order_count,
    ROUND(SUM(sales_per_customer)::numeric, 2) as total_revenue,
    ROUND(AVG(sales_per_customer)::numeric, 2) as avg_order_value,
    ROUND(SUM(benefit_per_order)::numeric, 2) as total_profit,
    ROUND(AVG(benefit_per_order)::numeric, 2) as avg_profit_per_order
FROM orders
GROUP BY late_delivery_risk
ORDER BY delivery_status;
"""

cursor.execute(query)
df_result = pd.DataFrame(cursor.fetchall(), columns=[desc[0] for desc in cursor.description])

print("="*70)
print("QUERY 10: REVENUE IMPACT BY DELIVERY STATUS")
print("="*70)
print(df_result.to_string(index=False))
print()

QUERY 10: REVENUE IMPACT BY DELIVERY STATUS
Empty DataFrame
Columns: [delivery_status, order_count, total_revenue, avg_order_value, total_profit, avg_profit_per_order]
Index: []



In [28]:
query = """
-- Query 11: Worst Performing State-Category Combinations
SELECT 
    customer_state,
    category_name,
    COUNT(*) as order_count,
    SUM(CASE WHEN late_delivery_risk = 1 THEN 1 ELSE 0 END) as late_count,
    ROUND(100.0 * SUM(CASE WHEN late_delivery_risk = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) as late_rate_percent,
    ROUND(AVG(days_for_shipping_real - days_for_shipment_scheduled), 2) as avg_delay_days
FROM orders
WHERE customer_state IN (
    SELECT customer_state FROM orders 
    GROUP BY customer_state 
    ORDER BY COUNT(*) DESC LIMIT 5
)
GROUP BY customer_state, category_name
HAVING COUNT(*) >= 50
ORDER BY late_rate_percent DESC
LIMIT 15;
"""

cursor.execute(query)
df_result = pd.DataFrame(cursor.fetchall(), columns=[desc[0] for desc in cursor.description])

print("="*70)
print("QUERY 11: WORST STATE-CATEGORY COMBINATIONS")
print("="*70)
print(df_result.to_string(index=False))
print()

QUERY 11: WORST STATE-CATEGORY COMBINATIONS
Empty DataFrame
Columns: [customer_state, category_name, order_count, late_count, late_rate_percent, avg_delay_days]
Index: []



In [29]:
query = """
-- Query 12: Delay Magnitude Analysis (CORRECTED)
SELECT 
    CASE 
        WHEN (days_for_shipping_real - days_for_shipment_scheduled) < 0 THEN 'Early (< 0 days)'
        WHEN (days_for_shipping_real - days_for_shipment_scheduled) = 0 THEN 'On-Time (0 days)'
        WHEN (days_for_shipping_real - days_for_shipment_scheduled) <= 1 THEN 'Minor (1 day)'
        WHEN (days_for_shipping_real - days_for_shipment_scheduled) <= 3 THEN 'Moderate (2-3 days)'
        ELSE 'Severe (4+ days)'
    END as delay_category,
    COUNT(*) as order_count,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) as percentage,
    ROUND(SUM(sales_per_customer)::numeric, 2) as total_revenue
FROM orders
GROUP BY 
    CASE 
        WHEN (days_for_shipping_real - days_for_shipment_scheduled) < 0 THEN 'Early (< 0 days)'
        WHEN (days_for_shipping_real - days_for_shipment_scheduled) = 0 THEN 'On-Time (0 days)'
        WHEN (days_for_shipping_real - days_for_shipment_scheduled) <= 1 THEN 'Minor (1 day)'
        WHEN (days_for_shipping_real - days_for_shipment_scheduled) <= 3 THEN 'Moderate (2-3 days)'
        ELSE 'Severe (4+ days)'
    END
ORDER BY 
    CASE 
        WHEN 
            CASE 
                WHEN (days_for_shipping_real - days_for_shipment_scheduled) < 0 THEN 'Early (< 0 days)'
                WHEN (days_for_shipping_real - days_for_shipment_scheduled) = 0 THEN 'On-Time (0 days)'
                WHEN (days_for_shipping_real - days_for_shipment_scheduled) <= 1 THEN 'Minor (1 day)'
                WHEN (days_for_shipping_real - days_for_shipment_scheduled) <= 3 THEN 'Moderate (2-3 days)'
                ELSE 'Severe (4+ days)'
            END = 'Early (< 0 days)' THEN 1
        WHEN 
            CASE 
                WHEN (days_for_shipping_real - days_for_shipment_scheduled) < 0 THEN 'Early (< 0 days)'
                WHEN (days_for_shipping_real - days_for_shipment_scheduled) = 0 THEN 'On-Time (0 days)'
                WHEN (days_for_shipping_real - days_for_shipment_scheduled) <= 1 THEN 'Minor (1 day)'
                WHEN (days_for_shipping_real - days_for_shipment_scheduled) <= 3 THEN 'Moderate (2-3 days)'
                ELSE 'Severe (4+ days)'
            END = 'On-Time (0 days)' THEN 2
        WHEN 
            CASE 
                WHEN (days_for_shipping_real - days_for_shipment_scheduled) < 0 THEN 'Early (< 0 days)'
                WHEN (days_for_shipping_real - days_for_shipment_scheduled) = 0 THEN 'On-Time (0 days)'
                WHEN (days_for_shipping_real - days_for_shipment_scheduled) <= 1 THEN 'Minor (1 day)'
                WHEN (days_for_shipping_real - days_for_shipment_scheduled) <= 3 THEN 'Moderate (2-3 days)'
                ELSE 'Severe (4+ days)'
            END = 'Minor (1 day)' THEN 3
        WHEN 
            CASE 
                WHEN (days_for_shipping_real - days_for_shipment_scheduled) < 0 THEN 'Early (< 0 days)'
                WHEN (days_for_shipping_real - days_for_shipment_scheduled) = 0 THEN 'On-Time (0 days)'
                WHEN (days_for_shipping_real - days_for_shipment_scheduled) <= 1 THEN 'Minor (1 day)'
                WHEN (days_for_shipping_real - days_for_shipment_scheduled) <= 3 THEN 'Moderate (2-3 days)'
                ELSE 'Severe (4+ days)'
            END = 'Moderate (2-3 days)' THEN 4
        ELSE 5
    END;
"""

cursor.execute(query)
df_result = pd.DataFrame(cursor.fetchall(), columns=[desc[0] for desc in cursor.description])

print("="*70)
print("QUERY 12: DELAY MAGNITUDE ANALYSIS")
print("="*70)
print(df_result.to_string(index=False))
print()

QUERY 12: DELAY MAGNITUDE ANALYSIS
Empty DataFrame
Columns: [delay_category, order_count, percentage, total_revenue]
Index: []



In [30]:
query = """
-- Query 13: Top 10 Best Performing Shipping Mode-Region Pairs
SELECT 
    shipping_mode,
    customer_state,
    COUNT(*) as order_count,
    ROUND(100.0 * SUM(CASE WHEN late_delivery_risk = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) as late_rate_percent,
    ROUND(AVG(days_for_shipping_real - days_for_shipment_scheduled), 2) as avg_delay_days,
    ROUND(AVG(sales_per_customer)::numeric, 2) as avg_order_value
FROM orders
WHERE customer_state IN (
    SELECT customer_state FROM orders 
    GROUP BY customer_state 
    ORDER BY COUNT(*) DESC LIMIT 5
)
GROUP BY shipping_mode, customer_state
HAVING COUNT(*) >= 30
ORDER BY late_rate_percent ASC
LIMIT 10;
"""

cursor.execute(query)
df_result = pd.DataFrame(cursor.fetchall(), columns=[desc[0] for desc in cursor.description])

print("="*70)
print("QUERY 13: TOP 10 BEST PERFORMING SHIPPING-REGION PAIRS")
print("="*70)
print(df_result.to_string(index=False))
print()

QUERY 13: TOP 10 BEST PERFORMING SHIPPING-REGION PAIRS
Empty DataFrame
Columns: [shipping_mode, customer_state, order_count, late_rate_percent, avg_delay_days, avg_order_value]
Index: []



In [34]:
# RESET TRANSACTION
try:
    connection.rollback()
    print("✓ Transaction rolled back successfully")
except:
    pass

# Verify connection is clean
cursor.execute("SELECT 1;")
print("✓ Connection is healthy\n")

✓ Transaction rolled back successfully
✓ Connection is healthy



In [35]:
query = """
-- Query 14: Order Value vs Delivery Performance (FIXED)
SELECT 
    CASE 
        WHEN sales_per_customer < 500 THEN 'Low (<$500)'
        WHEN sales_per_customer < 1000 THEN 'Medium ($500-1K)'
        WHEN sales_per_customer < 2000 THEN 'High ($1K-2K)'
        ELSE 'Very High (>$2K)'
    END as value_segment,
    COUNT(*) as order_count,
    ROUND(100.0 * SUM(CASE WHEN late_delivery_risk = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) as late_rate_percent,
    ROUND(AVG(days_for_shipping_real - days_for_shipment_scheduled), 2) as avg_delay_days,
    ROUND(AVG(sales_per_customer)::numeric, 2) as avg_order_value
FROM orders
GROUP BY 
    CASE 
        WHEN sales_per_customer < 500 THEN 'Low (<$500)'
        WHEN sales_per_customer < 1000 THEN 'Medium ($500-1K)'
        WHEN sales_per_customer < 2000 THEN 'High ($1K-2K)'
        ELSE 'Very High (>$2K)'
    END
ORDER BY COUNT(*) DESC;
"""

cursor.execute(query)
df_result = pd.DataFrame(cursor.fetchall(), columns=[desc[0] for desc in cursor.description])

print("="*70)
print("QUERY 14: ORDER VALUE vs DELIVERY PERFORMANCE")
print("="*70)
print(df_result.to_string(index=False))
print()

QUERY 14: ORDER VALUE vs DELIVERY PERFORMANCE
Empty DataFrame
Columns: [value_segment, order_count, late_rate_percent, avg_delay_days, avg_order_value]
Index: []



In [36]:
query = """
-- Query 15: Profitability Analysis
SELECT 
    CASE 
        WHEN late_delivery_risk = 1 THEN 'Late Delivery'
        ELSE 'On-Time Delivery'
    END as delivery_status,
    COUNT(*) as order_count,
    ROUND(SUM(sales_per_customer)::numeric, 2) as total_revenue,
    ROUND(SUM(benefit_per_order)::numeric, 2) as total_profit,
    ROUND((SUM(benefit_per_order)::numeric / SUM(sales_per_customer)::numeric * 100), 2) as profit_margin_percent,
    ROUND(AVG(benefit_per_order)::numeric, 2) as avg_profit_per_order
FROM orders
GROUP BY late_delivery_risk
ORDER BY delivery_status DESC;
"""

cursor.execute(query)
df_result = pd.DataFrame(cursor.fetchall(), columns=[desc[0] for desc in cursor.description])

print("="*70)
print("QUERY 15: PROFITABILITY ANALYSIS BY DELIVERY STATUS")
print("="*70)
print(df_result.to_string(index=False))
print()

QUERY 15: PROFITABILITY ANALYSIS BY DELIVERY STATUS
Empty DataFrame
Columns: [delivery_status, order_count, total_revenue, total_profit, profit_margin_percent, avg_profit_per_order]
Index: []



In [39]:
try:
    connection.rollback()
    print("✓ Transaction rolled back successfully")
except:
    pass

# Verify connection is clean
cursor.execute("SELECT 1;")
print("✓ Connection is healthy\n")

✓ Transaction rolled back successfully
✓ Connection is healthy



In [40]:
query = """
-- Query 16: Late Orders by Shipping Mode
SELECT 
    shipping_mode,
    COUNT(*) as order_count,
    SUM(CASE WHEN late_delivery_risk = 1 THEN 1 ELSE 0 END) as late_count,
    ROUND(100.0 * SUM(CASE WHEN late_delivery_risk = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) as late_rate_percent,
    ROUND(AVG(days_for_shipping_real - days_for_shipment_scheduled), 2) as avg_delay_days,
    ROUND(SUM(sales_per_customer)::numeric, 2) as total_revenue
FROM orders
GROUP BY shipping_mode
ORDER BY late_rate_percent DESC;
"""

cursor.execute(query)
df_result = pd.DataFrame(cursor.fetchall(), columns=[desc[0] for desc in cursor.description])

print("="*70)
print("QUERY 16: LATE ORDERS BY SHIPPING MODE")
print("="*70)
print(df_result.to_string(index=False))
print()

QUERY 16: LATE ORDERS BY SHIPPING MODE
Empty DataFrame
Columns: [shipping_mode, order_count, late_count, late_rate_percent, avg_delay_days, total_revenue]
Index: []



In [41]:
query = """
-- Query 17: Carrier Performance Ranking (Using Shipping Mode as Proxy)
WITH carrier_metrics AS (
    SELECT 
        shipping_mode as carrier,
        COUNT(*) as order_count,
        SUM(CASE WHEN late_delivery_risk = 1 THEN 1 ELSE 0 END) as late_count,
        ROUND(100.0 * SUM(CASE WHEN late_delivery_risk = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) as late_rate_percent,
        ROUND(AVG(days_for_shipping_real - days_for_shipment_scheduled), 2) as avg_delay_days,
        ROUND(AVG(sales_per_customer)::numeric, 2) as avg_order_value
    FROM orders
    GROUP BY shipping_mode
)
SELECT 
    carrier,
    order_count,
    late_count,
    late_rate_percent,
    avg_delay_days,
    avg_order_value,
    RANK() OVER (ORDER BY late_rate_percent ASC) as performance_rank
FROM carrier_metrics
ORDER BY performance_rank;
"""

cursor.execute(query)
df_result = pd.DataFrame(cursor.fetchall(), columns=[desc[0] for desc in cursor.description])

print("="*70)
print("QUERY 17: CARRIER PERFORMANCE RANKING")
print("="*70)
print(df_result.to_string(index=False))
print()

QUERY 17: CARRIER PERFORMANCE RANKING
Empty DataFrame
Columns: [carrier, order_count, late_count, late_rate_percent, avg_delay_days, avg_order_value, performance_rank]
Index: []



In [42]:
query = """
-- Query 18: Regional Benchmark Comparison
WITH regional_metrics AS (
    SELECT 
        customer_state,
        COUNT(*) as order_count,
        ROUND(100.0 * SUM(CASE WHEN late_delivery_risk = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) as late_rate_percent,
        ROUND(AVG(days_for_shipping_real - days_for_shipment_scheduled), 2) as avg_delay_days
    FROM orders
    GROUP BY customer_state
    HAVING COUNT(*) > 100
),
national_avg AS (
    SELECT 
        ROUND(100.0 * SUM(CASE WHEN late_delivery_risk = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) as national_late_rate,
        ROUND(AVG(days_for_shipping_real - days_for_shipment_scheduled), 2) as national_avg_delay
    FROM orders
)
SELECT 
    customer_state,
    order_count,
    late_rate_percent,
    (SELECT national_late_rate FROM national_avg) as national_benchmark,
    (late_rate_percent - (SELECT national_late_rate FROM national_avg)) as variance_from_benchmark,
    avg_delay_days
FROM regional_metrics
ORDER BY variance_from_benchmark DESC;
"""

cursor.execute(query)
df_result = pd.DataFrame(cursor.fetchall(), columns=[desc[0] for desc in cursor.description])

print("="*70)
print("QUERY 18: REGIONAL BENCHMARK COMPARISON")
print("="*70)
print(df_result.to_string(index=False))
print()

QUERY 18: REGIONAL BENCHMARK COMPARISON
Empty DataFrame
Columns: [customer_state, order_count, late_rate_percent, national_benchmark, variance_from_benchmark, avg_delay_days]
Index: []



In [43]:
query = """
-- Query 19: Customer Retention Risk Analysis
SELECT 
    customer_segment,
    COUNT(*) as total_orders,
    SUM(CASE WHEN late_delivery_risk = 1 THEN 1 ELSE 0 END) as late_orders,
    ROUND(100.0 * SUM(CASE WHEN late_delivery_risk = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) as late_rate_percent,
    ROUND(SUM(sales_per_customer)::numeric, 2) as revenue_at_risk,
    ROUND(SUM(CASE WHEN late_delivery_risk = 1 THEN sales_per_customer ELSE 0 END)::numeric, 2) as late_order_revenue
FROM orders
GROUP BY customer_segment
ORDER BY late_rate_percent DESC;
"""

cursor.execute(query)
df_result = pd.DataFrame(cursor.fetchall(), columns=[desc[0] for desc in cursor.description])

print("="*70)
print("QUERY 19: CUSTOMER RETENTION RISK ANALYSIS")
print("="*70)
print(df_result.to_string(index=False))
print()

QUERY 19: CUSTOMER RETENTION RISK ANALYSIS
Empty DataFrame
Columns: [customer_segment, total_orders, late_orders, late_rate_percent, revenue_at_risk, late_order_revenue]
Index: []



In [44]:
query = """
-- Query 20: Cost-Benefit Analysis (Assuming shipping mode has cost implications)
SELECT 
    shipping_mode,
    COUNT(*) as order_count,
    ROUND(AVG(sales_per_customer)::numeric, 2) as avg_revenue,
    ROUND(100.0 * SUM(CASE WHEN late_delivery_risk = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) as late_rate_percent,
    CASE 
        WHEN shipping_mode = 'Same Day' THEN 50
        WHEN shipping_mode = 'First Class' THEN 25
        WHEN shipping_mode = 'Second Class' THEN 15
        ELSE 10
    END as estimated_cost,
    CASE 
        WHEN shipping_mode = 'Same Day' THEN 50
        WHEN shipping_mode = 'First Class' THEN 25
        WHEN shipping_mode = 'Second Class' THEN 15
        ELSE 10
    END * COUNT(*) as total_estimated_cost,
    ROUND((ROUND(AVG(sales_per_customer)::numeric, 2) * COUNT(*)) - 
        (CASE 
            WHEN shipping_mode = 'Same Day' THEN 50
            WHEN shipping_mode = 'First Class' THEN 25
            WHEN shipping_mode = 'Second Class' THEN 15
            ELSE 10
        END * COUNT(*)), 2) as net_profit
FROM orders
GROUP BY shipping_mode
ORDER BY net_profit DESC;
"""

cursor.execute(query)
df_result = pd.DataFrame(cursor.fetchall(), columns=[desc[0] for desc in cursor.description])

print("="*70)
print("QUERY 20: COST-BENEFIT ANALYSIS OF SHIPPING METHODS")
print("="*70)
print(df_result.to_string(index=False))
print()

QUERY 20: COST-BENEFIT ANALYSIS OF SHIPPING METHODS
Empty DataFrame
Columns: [shipping_mode, order_count, avg_revenue, late_rate_percent, estimated_cost, total_estimated_cost, net_profit]
Index: []



In [45]:
connection.close()
print("\n✓ All queries executed successfully!")
print("✓ PostgreSQL connection closed")


✓ All queries executed successfully!
✓ PostgreSQL connection closed
